# Model verification and structural stability

This notebook tests whether the structure discovered on a training dataset is preserved on a held-out test dataset.

The workflow is intentionally descriptive and diagnostic:

- discover structure on the training data
- evaluate the same structure logic on the test data
- compare the discovered patterns
- interpret material differences as a drift or stability signal rather than as a hard failure condition

This notebook uses the prepared numeric dataset in `data/model_ready_numeric_data.csv`, where the upstream cleaning and numeric featurization steps are assumed to be complete.


In [1]:
from pathlib import Path
import pandas as pd

candidate_roots = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
]
project_root = next((root for root in candidate_roots if (root / 'data' / 'model_ready_numeric_data.csv').exists()), Path.cwd())
data_path = project_root / 'data' / 'model_ready_numeric_data.csv'
print(f'Project root: {project_root}')
print(f'Data path: {data_path}')
df = pd.read_csv(data_path)
print(df.shape)
print(df['dataset_partition'].value_counts(dropna=False).to_dict())


Project root: /home/rajiv/programming/structure_identification_for_CS_data
Data path: /home/rajiv/programming/structure_identification_for_CS_data/data/model_ready_numeric_data.csv
(116341, 22)
{'active': 55845, 'train': 48396, 'validation': 6050, 'test': 6050}


In [2]:
from structure_id import structure_id

train = df[df['dataset_partition'].eq('train')].drop(columns=['dataset_partition']).copy()
test = df[df['dataset_partition'].eq('test')].drop(columns=['dataset_partition']).copy()

train_result = structure_id(
    data=train,
    target='loan_status_r',
    task_type='classification',
    structural_covariates=[],
)

test_result = structure_id(
    data=test,
    target='loan_status_r',
    task_type='classification',
    structural_covariates=[],
)

print(f'Train rows: {len(train)}')
print(f'Test rows: {len(test)}')
print(f'Train explicit blocks: {len(train_result.explicit_column_structures)}')
print(f'Test explicit blocks: {len(test_result.explicit_column_structures)}')
print(f'Train row clusters: {len(train_result.explicit_row_clusters)}')
print(f'Test row clusters: {len(test_result.explicit_row_clusters)}')
print(f'Train latent break: {train_result.latent_break.variable_name if train_result.latent_break else None}')
print(f'Test latent break: {test_result.latent_break.variable_name if test_result.latent_break else None}')
print(f'Train latent significance: {train_result.latent_break.significant if train_result.latent_break else None}')
print(f'Test latent significance: {test_result.latent_break.significant if test_result.latent_break else None}')

print('=== TRAIN SUMMARY ===')
print(train_result.summary())
print('=== TEST SUMMARY ===')
print(test_result.summary())


Train rows: 48396
Test rows: 6050
Train explicit blocks: 0
Test explicit blocks: 0
Train row clusters: 159
Test row clusters: 13
Train latent break: projectcounty_rcs_te
Test latent break: projectcounty_rcs_te
Train latent significance: True
Test latent significance: False
=== TRAIN SUMMARY ===
Structure Identification Summary
Target: loan_status_r
Task type: classification
Provisional candidates: 16
Selected candidates: 17
Explicit column blocks: 0
Explicit row clusters: 159
Instance structure summary:
  noise=15660
  cluster_0=45
  cluster_1=55
  cluster_2=141
  cluster_3=144
  cluster_4=254
  cluster_5=299
  cluster_6=721
  cluster_7=30
  cluster_8=30
  cluster_9=187
  cluster_10=141
  cluster_11=4296
  cluster_12=2623
  cluster_13=25
  cluster_14=20
  cluster_15=40
  cluster_16=27
  cluster_17=12131
  cluster_18=135
  cluster_19=35
  cluster_20=182
  cluster_21=47
  cluster_22=56
  cluster_23=50
  cluster_24=34
  cluster_25=18
  cluster_26=30
  cluster_27=70
  cluster_28=35
  clust

## Interpretation

The training dataset reveals a richer structure than the test dataset. In this example:

- the training data produces substantially more row clusters
- the training data shows a statistically significant latent structural break
- the test data shows fewer discovered clusters and no significant latent break

This is the expected use case for the package: a discovered structure on the training or analysis data is compared against a held-out dataset to check whether the same structure remains stable. When the structure differs materially, the first question to investigate is drift or a change in the underlying operating population.


In [3]:
from collections import Counter

train_values = train_result.diagnostics.get('structure_id_values', {})
test_values = test_result.diagnostics.get('structure_id_values', {})

def summarize_structure(values):
    counts = Counter(str(v) for v in values.values())
    noise = counts.get('noise', 0)
    cluster_counts = sorted((int(k), v) for k, v in counts.items() if k != 'noise')
    return {'noise': noise, 'clusters': cluster_counts}

train_summary = summarize_structure(train_values)
test_summary = summarize_structure(test_values)

print('Train structure counts:', train_summary)
print('Test structure counts:', test_summary)
print('Train latent break p-value:', train_result.latent_break.adjusted_p_value if train_result.latent_break else None)
print('Test latent break p-value:', test_result.latent_break.adjusted_p_value if test_result.latent_break else None)


Train structure counts: {'noise': 15660, 'clusters': [(0, 45), (1, 55), (2, 141), (3, 144), (4, 254), (5, 299), (6, 721), (7, 30), (8, 30), (9, 187), (10, 141), (11, 4296), (12, 2623), (13, 25), (14, 20), (15, 40), (16, 27), (17, 12131), (18, 135), (19, 35), (20, 182), (21, 47), (22, 56), (23, 50), (24, 34), (25, 18), (26, 30), (27, 70), (28, 35), (29, 23), (30, 15), (31, 40), (32, 52), (33, 24), (34, 23), (35, 138), (36, 16), (37, 15), (38, 51), (39, 52), (40, 36), (41, 15), (42, 24), (43, 40), (44, 34), (45, 21), (46, 16), (47, 21), (48, 20), (49, 18), (50, 15), (51, 21), (52, 25), (53, 27), (54, 38), (55, 49), (56, 5250), (57, 75), (58, 21), (59, 46), (60, 71), (61, 33), (62, 73), (63, 26), (64, 35), (65, 23), (66, 58), (67, 25), (68, 18), (69, 21), (70, 24), (71, 213), (72, 42), (73, 29), (74, 15), (75, 39), (76, 39), (77, 57), (78, 15), (79, 126), (80, 38), (81, 43), (82, 17), (83, 31), (84, 43), (85, 166), (86, 78), (87, 31), (88, 87), (89, 120), (90, 48), (91, 21), (92, 88), (93